# CityCab Rides — Mini Project Challenge

**Analyst:** Jordan Lee  |  **Role:** Lead Junior Analyst  

**Brief:** Management handed us a messy ride export and asked for a complete analysis to improve operations. Workflow: **Inspect → Clean → Explore → Visualize → SQL → Insights → Report**.

> Before submitting: **Kernel → Restart & Run All**.

In [1]:
import os, sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
os.makedirs("charts", exist_ok=True)

## Step 1 — Inspect (Day 4)

In [2]:
rides = pd.DataFrame({
    "RideID":   [1, 2, 3, 3, 5, 6, 7, 8, 9, 10],
    "Driver":   ["  ravi", "ASHA", "imran", "imran", "Divya",
                 "KARAN", "meena ", "Sahil", "tara", "asha"],
    "City":     ["Pune", "mumbai", "Pune", "Pune", "Delhi",
                 "Mumbai", "delhi", "Pune", "Mumbai", "mumbai"],
    "Distance": [5.2, 12.0, 3.5, 3.5, 8.0, 2.1, -4.0, 15.0, 6.5, 200.0],
    "Fare":     [120, 300, np.nan, 90, 200, 60, 110, 380, 160, 5000],
    "RideDate": ["2026-05-01", "2026-05-01", "2026-05-02", "2026-05-02", "2026-05-03",
                 "2026-05-04", "2026-05-05", "2026-05-06", "2026-05-07", "2026-05-08"],
})
rides.head()

,RideID,Driver,City,Distance,Fare,RideDate
0,1,ravi,Pune,5.2,120.0,2026-05-01
1,2,ASHA,mumbai,12.0,300.0,2026-05-01
2,3,imran,Pune,3.5,NaN,2026-05-02
3,3,imran,Pune,3.5,90.0,2026-05-02
4,5,Divya,Delhi,8.0,200.0,2026-05-03


In [3]:
print("Shape:", rides.shape)
rides.describe()

Shape: (10, 6)


,RideID,Distance,Fare
count,10.000000,10.000000,9.000000
mean,5.400000,25.180000,713.333333
std,3.098387,61.650337,1610.846051
min,1.000000,-4.000000,60.000000
25%,3.000000,3.500000,110.000000
50%,5.500000,5.850000,160.000000
75%,7.750000,11.000000,300.000000
max,10.000000,200.000000,5000.000000


**Red flags:** RideID 3 duplicated, 1 missing Fare, Distance −4.0 (impossible), Distance 200 & Fare 5000 (outliers), mixed-case text, RideDate stored as text.

## Step 2 — Clean (Day 5)
The two copies of RideID 3 differ (one Fare is NaN), so we dedupe on the **business key** and then median-fill the remaining missing fare.

In [4]:
df = rides.copy()
df = df.drop_duplicates(subset=["RideID"]).reset_index(drop=True)   # duplicate RideID 3
df["RideDate"] = pd.to_datetime(df["RideDate"])                     # fix type
df["Driver"] = df["Driver"].str.strip().str.title()                # text
df["City"] = df["City"].str.strip().str.title()

# invalid distance (-4) -> median of valid distances
df.loc[df["Distance"] <= 0, "Distance"] = df.loc[df["Distance"] > 0, "Distance"].median()

# outlier distance (200 km) -> IQR flag -> median fill
Q1d, Q3d = df["Distance"].quantile([0.25, 0.75]); upper_d = Q3d + 1.5*(Q3d-Q1d)
df.loc[df["Distance"] > upper_d, "Distance"] = np.nan
df["Distance"] = df["Distance"].fillna(df["Distance"].median())

# outlier fare (5000) + missing fare -> IQR flag -> median fill (both)
Q1f, Q3f = df["Fare"].quantile([0.25, 0.75]); upper_f = Q3f + 1.5*(Q3f-Q1f)
df.loc[df["Fare"] > upper_f, "Fare"] = np.nan
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

assert df.isnull().sum().sum() == 0 and df.duplicated(subset=["RideID"]).sum() == 0
assert (df["Distance"] > 0).all() and (df["Fare"] > 0).all()
print("Clean shape:", df.shape)
df.to_csv("citycab_clean.csv", index=False)
df

Clean shape: (9, 6)


,RideID,Driver,City,Distance,Fare,RideDate
0,1,Ravi,Pune,5.200,120.0,2026-05-01
1,2,Asha,Mumbai,12.000,300.0,2026-05-01
2,3,Imran,Pune,3.500,160.0,2026-05-02
3,5,Divya,Delhi,8.000,200.0,2026-05-03
4,6,Karan,Mumbai,2.100,60.0,2026-05-04
5,7,Meena,Delhi,7.250,110.0,2026-05-05
6,8,Sahil,Pune,15.000,380.0,2026-05-06
7,9,Tara,Mumbai,6.500,160.0,2026-05-07
8,10,Asha,Mumbai,6.875,160.0,2026-05-08


## Step 3 — Explore (Day 6)

In [5]:
print("Average fare:", round(df["Fare"].mean(), 2))
print("Average distance:", round(df["Distance"].mean(), 2))
fare_by_city = df.groupby("City")["Fare"].sum().sort_values(ascending=False)
print("\nTotal fare by city:\n", fare_by_city, sep="")
print("\nCorrelation (Distance vs Fare):", round(df["Distance"].corr(df["Fare"]), 2))

Average fare: 183.33
Average distance: 7.38

Total fare by city:
City
Mumbai    680.0
Pune      660.0
Delhi     310.0
Name: Fare, dtype: float64

Correlation (Distance vs Fare): 0.93


**Read-out:** Mumbai (₹680) and Pune (₹660) are the top markets. Distance↔Fare correlation **+0.93** — longer rides cost more (a positive, intuitively causal relationship).

## Step 4 — Visualize (Days 7 & 8)
Bar, histogram, box plot, scatter, heatmap — saved to `charts/` at dpi=300.

In [6]:
def save(name):
    plt.tight_layout(); plt.savefig(f"charts/{name}", dpi=300)

plt.figure(); sns.barplot(x=fare_by_city.index, y=fare_by_city.values)
plt.title("Total Fare by City"); plt.xlabel("City"); plt.ylabel("Total Fare (Rs.)")
save("01_fare_by_city.png"); plt.show()

C:\Users\KOONASAISIRI\AppData\Local\Temp\ipykernel_17156\4044850166.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save("01_fare_by_city.png"); plt.show()


In [7]:
plt.figure(); sns.histplot(df["Fare"], bins=6, kde=True)
plt.title("Distribution of Fares"); plt.xlabel("Fare (Rs.)"); plt.ylabel("Count")
save("02_fare_distribution.png"); plt.show()

C:\Users\KOONASAISIRI\AppData\Local\Temp\ipykernel_17156\3564425874.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save("02_fare_distribution.png"); plt.show()


In [8]:
plt.figure(); sns.boxplot(y=df["Fare"])
plt.title("Fare Spread (Box Plot)"); plt.ylabel("Fare (Rs.)")
save("03_fare_box.png"); plt.show()

C:\Users\KOONASAISIRI\AppData\Local\Temp\ipykernel_17156\1050495244.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save("03_fare_box.png"); plt.show()


In [9]:
plt.figure(); sns.scatterplot(data=df, x="Distance", y="Fare", hue="City", s=120)
plt.title("Distance vs Fare"); plt.xlabel("Distance (km)"); plt.ylabel("Fare (Rs.)")
save("04_distance_vs_fare.png"); plt.show()

C:\Users\KOONASAISIRI\AppData\Local\Temp\ipykernel_17156\3635473959.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save("04_distance_vs_fare.png"); plt.show()


In [10]:
plt.figure(); sns.heatmap(df[["Distance", "Fare"]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
save("05_correlation_heatmap.png"); plt.show()

C:\Users\KOONASAISIRI\AppData\Local\Temp\ipykernel_17156\157355172.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  save("05_correlation_heatmap.png"); plt.show()


## Step 5 — SQL (Day 9)

In [11]:
conn = sqlite3.connect(":memory:")
df.to_sql("rides", conn, index=False, if_exists="replace")

print("Q1 — Total fare by city:")
display(pd.read_sql("SELECT City, SUM(Fare) AS TotalFare FROM rides "
                    "GROUP BY City ORDER BY TotalFare DESC", conn))

print("Q2 — Rides above Rs.150:")
display(pd.read_sql("SELECT Driver, City, Fare FROM rides "
                    "WHERE Fare > 150 ORDER BY Fare DESC", conn))

print("Q3 — Pune ride list:")
display(pd.read_sql("SELECT Driver, Distance, Fare FROM rides WHERE City = 'Pune'", conn))

conn.close()

Q1 — Total fare by city:


,City,TotalFare
0,Mumbai,680.0
1,Pune,660.0
2,Delhi,310.0


Q2 — Rides above Rs.150:


,Driver,City,Fare
0,Sahil,Pune,380.0
1,Asha,Mumbai,300.0
2,Divya,Delhi,200.0
3,Imran,Pune,160.0
4,Tara,Mumbai,160.0
5,Asha,Mumbai,160.0


Q3 — Pune ride list:


,Driver,Distance,Fare
0,Ravi,5.2,120.0
1,Imran,3.5,160.0
2,Sahil,15.0,380.0


## Step 6 — Insights (the Insight Ladder)

| Observation | Finding | Insight | Recommendation |
|---|---|---|---|
| Mumbai has the highest total fare (₹680) | Busiest / highest-earning market | Demand is concentrated in Mumbai (& Pune) | Allocate more drivers to Mumbai & Pune at peak times |
| Distance↔Fare correlation +0.93 | Longer rides cost more | Fare scales with distance (intuitively causal) | Use the distance–fare line to flag mispriced rides |
| A 200 km ride & ₹5,000 fare appeared | Impossible values reached the export | Input validation is weak | Add range checks on distance & fare at entry |

## Step 7 — Report
Full eight-section report in **`EDA_REPORT.md`**. Bottom line: **Mumbai and Pune drive fares**, **fare scales with distance (+0.93)**, and the 200 km / ₹5,000 errors expose a data-quality gap. Rebalance drivers, validate input, and monitor the distance–fare line.